In [7]:
import os
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
BASE_DATA_PATH = os.path.join(NOTEBOOK_DIR, "..", "DSA4264 Data")

# Input paths
COURSES_DIR = os.path.join(BASE_DATA_PATH, "NUS-SMU-SUTD Courses")
JOBS_PARQUET = os.path.join(BASE_DATA_PATH, "processed_jobs_dual_embeddings.parquet")

# Course CSV files
NUS_COURSES_CSV = os.path.join(COURSES_DIR, "nus_courses.csv")
SMU_COURSES_CSV = os.path.join(COURSES_DIR, "smu_courses.csv")
SUTD_COURSES_CSV = os.path.join(COURSES_DIR, "sutd_courses.csv")

# Output directories
OUTPUT_DIR = os.path.join(NOTEBOOK_DIR, "bertopic_visualizations_global")
# CHANGED: Output job matches to DSA4264 Data/Jobs by courses/ (overwrites for zoom_in_analysis)
UNIVERSITY_JOB_MATCHES_DIR = os.path.join(BASE_DATA_PATH, "Jobs by courses")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(UNIVERSITY_JOB_MATCHES_DIR, exist_ok=True)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Data path: {BASE_DATA_PATH}")
print(f"Jobs parquet: {JOBS_PARQUET}")
print(f"Courses directory: {COURSES_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Job matches output: {UNIVERSITY_JOB_MATCHES_DIR}")



Notebook directory: /Users/teresaliau/dsa4264/notebooks
Data path: /Users/teresaliau/dsa4264/notebooks/../DSA4264 Data
Jobs parquet: /Users/teresaliau/dsa4264/notebooks/../DSA4264 Data/processed_jobs_dual_embeddings.parquet
Courses directory: /Users/teresaliau/dsa4264/notebooks/../DSA4264 Data/NUS-SMU-SUTD Courses
Output directory: /Users/teresaliau/dsa4264/notebooks/bertopic_visualizations_global
Job matches output: /Users/teresaliau/dsa4264/notebooks/../DSA4264 Data/Jobs by courses


In [ ]:


os.makedirs(UNIVERSITY_JOB_MATCHES_DIR, exist_ok=True)

jobs_df = pd.read_parquet(os.path.join(NOTEBOOK_DIR, "jobs_with_topics.parquet"))
jobs_clustered = jobs_df[jobs_df["topic_id"] != -1].copy().reset_index(drop=True)

# Load existing BERTopic model
try:
    topic_model
except NameError:
    embedding_model = SentenceTransformer("all-mpnet-base-v2")
    topic_model = BERTopic.load(os.path.join(NOTEBOOK_DIR, "bertopic_global_model"), embedding_model=embedding_model)


def run_university_matching(course_csv_path, modules_parquet_path, university_prefix):
    print(f"\n===== Processing {university_prefix.upper()} =====")

    csv_df = pd.read_csv(course_csv_path)
    csv_df.columns = csv_df.columns.str.strip().str.lower()

    all_courses = csv_df["course"].dropna().unique()
    all_modules_df = pd.read_parquet(modules_parquet_path).copy()

    # assign topic clusters to modules
    module_texts = all_modules_df["description"].fillna("").tolist()
    module_topics, _ = topic_model.transform(module_texts)
    all_modules_df["assigned_topic_id"] = module_topics

    topic_info = topic_model.get_topic_info()
    topic_label_map = {
        row["Topic"]: ", ".join(row["Representation"][:3])
        for _, row in topic_info.iterrows()
    }
    all_modules_df["assigned_topic_label"] = (
        all_modules_df["assigned_topic_id"]
        .map(topic_label_map)
        .fillna("Outlier")
    )

    for course_name in all_courses:
        required_codes = (
            csv_df[csv_df["course"] == course_name]["code"]
            .dropna()
            .astype(str)
            .unique()
        )

        degree_modules_df = all_modules_df[
            all_modules_df["code"].astype(str).isin(required_codes)
        ].copy()

        if len(degree_modules_df) == 0:
            print(f"Skipping '{course_name}' - No modules found in embedded dataset.")
            continue

        topic_module_counts = (
            degree_modules_df[degree_modules_df["assigned_topic_id"] != -1]
            .groupby(["assigned_topic_id", "assigned_topic_label"])
            .size()
            .reset_index(name="module_count")
            .sort_values("module_count", ascending=False)
        )

        if len(topic_module_counts) == 0:
            print(f"Skipping '{course_name}' - Modules only matched with Outliers.")
            continue

        relevant_topic_ids = topic_module_counts["assigned_topic_id"].tolist()
        topic_weight_map = dict(
            zip(topic_module_counts["assigned_topic_id"], topic_module_counts["module_count"])
        )

        matched_jobs = jobs_clustered[
            jobs_clustered["topic_id"].isin(relevant_topic_ids)
        ].copy()

        if len(matched_jobs) == 0:
            print(f"Skipping '{course_name}' - No matched jobs found.")
            continue

        matched_jobs["topic_module_count"] = (
            matched_jobs["topic_id"].map(topic_weight_map).fillna(0)
        )

        # cosine similarity
        course_embeddings = np.stack(degree_modules_df["skill_embedding"].values)
        job_desc_embeddings = np.stack(matched_jobs["embedding_mpnet"].values)

        sim_matrix = cosine_similarity(course_embeddings, job_desc_embeddings)

        matched_jobs["avg_similarity"] = sim_matrix.mean(axis=0)
        matched_jobs["max_similarity"] = sim_matrix.max(axis=0)

        top_module_idx = sim_matrix.argmax(axis=0)

        if "title" in degree_modules_df.columns:
            matched_jobs["best_module"] = [
                f"{degree_modules_df.iloc[i]['code']} – {degree_modules_df.iloc[i]['title']}"
                for i in top_module_idx
            ]
        elif "type" in degree_modules_df.columns:
            matched_jobs["best_module"] = [
                f"{degree_modules_df.iloc[i]['code']} ({degree_modules_df.iloc[i]['type']})"
                for i in top_module_idx
            ]
        else:
            matched_jobs["best_module"] = [
                str(degree_modules_df.iloc[i]["code"])
                for i in top_module_idx
            ]

        max_module_count = topic_module_counts["module_count"].max()
        matched_jobs["combined_score"] = (
            0.4 * (matched_jobs["topic_module_count"] / max_module_count) +
            0.4 * matched_jobs["avg_similarity"] +
            0.2 * matched_jobs["max_similarity"]
        )

        safe_course = str(course_name).replace(" ", "_").replace("/", "_").replace("\\", "_")
        output_path = os.path.join(
            UNIVERSITY_JOB_MATCHES_DIR,
            f"{university_prefix}_{safe_course}_matches.csv"
        )

        output_cols = [
            "title",
            "combined_score",
            "topic_label",
            "topic_module_count",
            "avg_similarity",
            "max_similarity",
            "best_module",
            "description"
        ]
        if "companyName" in matched_jobs.columns:
            output_cols.insert(1, "companyName")
        output_cols = [c for c in output_cols if c in matched_jobs.columns]

        top_matches = matched_jobs.sort_values("combined_score", ascending=False)
        top_matches[output_cols].to_csv(output_path, index=False)

        top_job_title = str(top_matches.iloc[0]["title"])[:35] if len(top_matches) > 0 else "N/A"
        print(f"{course_name[:30]:<30} | {len(degree_modules_df):>3} modules | Top Job: {top_job_title}")


# Run for each university
run_university_matching(
    course_csv_path=NUS_COURSES_CSV,
    modules_parquet_path=os.path.join(COURSES_DIR, "10_nus_modules_embedded.parquet"),
    university_prefix="nus"
)

run_university_matching(
    course_csv_path=SMU_COURSES_CSV,
    modules_parquet_path=os.path.join(COURSES_DIR, "10_smu_modules_embedded.parquet"),
    university_prefix="smu"
)

run_university_matching(
    course_csv_path=SUTD_COURSES_CSV,
    modules_parquet_path=os.path.join(COURSES_DIR, "10_sutd_modules_embedded.parquet"),
    university_prefix="sutd"
)

print("\n✓ All job matches saved to:", UNIVERSITY_JOB_MATCHES_DIR)



===== Processing NUS =====
accountancy                    |  37 modules | Top Job: Accountant - ERP Systems / AR & AP 
biomedical_engineering         |  49 modules | Top Job: Research Fellow
business_analytics             |  98 modules | Top Job: AI Software Developer
chemical_engineering           |  76 modules | Top Job: Maintenance (Mechanical) Trainee - 
civil_engineering              |  77 modules | Top Job: Principal Structural Engineer
data_sci_analytics             |  56 modules | Top Job: Principal Data Scientist (Aerospace
industrial_design              |  34 modules | Top Job: Interior Designer (Hospitality)
information_systems            | 113 modules | Top Job: AI Software Developer
landscape_architecture         |  30 modules | Top Job: Interior Designer (Hospitality)
real_estate                    |  25 modules | Top Job: Director, Transactions Management

===== Processing SMU =====
economics                      |  71 modules | Top Job: Presidential Assistant Professor